# 01 - An?lisis Exploratorio del Dataset

Este notebook analiza el dataset antes del entrenamiento.

Objetivos:

- Contar im?genes por clase.
- Revisar Train / Validation / Test.
- Revisar dimensiones.
- Visualizar ejemplos.
- Visualizar las tres c?maras de una misma palta.


In [ ]:
from pathlib import Path
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt

# Ruta del dataset
RUTA_DATASET = Path(r"..\DATASET_PALTA")

EXTENSIONES = {".tif", ".tiff", ".png", ".jpg", ".jpeg", ".bmp"}

print("Ruta:", RUTA_DATASET.resolve())
print("Existe:", RUTA_DATASET.exists())


## Distribuci?n de im?genes

Se contar? autom?ticamente el n?mero de im?genes existente en cada conjunto y clase.


In [ ]:
registros = []

for conjunto in ["train", "validation", "test"]:

    ruta_conjunto = RUTA_DATASET / conjunto

    if not ruta_conjunto.exists():
        print(f"No encontrada: {ruta_conjunto}")
        continue

    for carpeta_clase in ruta_conjunto.iterdir():

        if not carpeta_clase.is_dir():
            continue

        archivos = [
            archivo
            for archivo in carpeta_clase.rglob("*")
            if archivo.is_file() and archivo.suffix.lower() in EXTENSIONES
        ]

        registros.append({
            "Conjunto": conjunto,
            "Clase": carpeta_clase.name,
            "Cantidad": len(archivos)
        })

tabla_distribucion = pd.DataFrame(registros)

display(tabla_distribucion)

if not tabla_distribucion.empty:

    tabla_grafico = tabla_distribucion.pivot(
        index="Conjunto",
        columns="Clase",
        values="Cantidad"
    ).fillna(0)

    ax = tabla_grafico.plot(
        kind="bar",
        figsize=(9, 5)
    )

    ax.set_title("Distribuci?n del dataset")
    ax.set_ylabel("N?mero de im?genes")
    ax.set_xlabel("Conjunto")

    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


## Dimensiones de las im?genes

Se revisan las resoluciones existentes para comprobar la consistencia del dataset.


In [ ]:
dimensiones = []

for conjunto in ["train", "validation", "test"]:

    ruta_conjunto = RUTA_DATASET / conjunto

    if not ruta_conjunto.exists():
        continue

    for archivo in ruta_conjunto.rglob("*"):

        if not archivo.is_file():
            continue

        if archivo.suffix.lower() not in EXTENSIONES:
            continue

        try:

            with Image.open(archivo) as imagen:

                dimensiones.append({
                    "Conjunto": conjunto,
                    "Archivo": archivo.name,
                    "Ancho": imagen.width,
                    "Alto": imagen.height
                })

        except Exception as error:

            print("Error:", archivo, error)

tabla_dimensiones = pd.DataFrame(dimensiones)

display(tabla_dimensiones.head())

if not tabla_dimensiones.empty:

    print("\nResoluciones m?s frecuentes:")

    display(
        tabla_dimensiones
        .groupby(["Ancho", "Alto"])
        .size()
        .reset_index(name="Cantidad")
        .sort_values("Cantidad", ascending=False)
        .head(10)
    )


## Visualizaci?n de una palta y las tres c?maras

Una imagen completa corresponde a una sola palta.

La imagen se divide ?nicamente para visualizar las tres c?maras; las franjas no se consideran muestras independientes.


In [ ]:
imagenes = [
    archivo
    for archivo in (RUTA_DATASET / "train").rglob("*")
    if archivo.is_file() and archivo.suffix.lower() in EXTENSIONES
]

if imagenes:

    archivo = imagenes[0]

    with Image.open(archivo) as imagen:

        imagen = imagen.convert("RGB")

        ancho, alto = imagen.size

        limite_1 = alto // 3
        limite_2 = 2 * alto // 3

        camara_1 = imagen.crop((0, 0, ancho, limite_1))
        camara_2 = imagen.crop((0, limite_1, ancho, limite_2))
        camara_3 = imagen.crop((0, limite_2, ancho, alto))

        print("Archivo:", archivo.name)
        print("Resoluci?n:", imagen.size)

        plt.figure(figsize=(14, 6))
        plt.imshow(imagen)
        plt.title("Imagen completa - una palta")
        plt.axis("off")
        plt.show()

        for numero, camara in enumerate(
            [camara_1, camara_2, camara_3],
            start=1
        ):

            plt.figure(figsize=(14, 3))
            plt.imshow(camara)
            plt.title(f"C?mara {numero}")
            plt.axis("off")
            plt.show()
